# w23_cpu1_pinns_video_audit_and_jrdb_ranker
Inputs: `shortpose-wave1-private-inputs` | Accelerator: `None` (Kaggle 4-vCPU Cloud) | Internet: `On`


In [ ]:
import hashlib, os, subprocess, sys, tempfile, zipfile
from pathlib import Path

WORK, MOUNT = Path('/kaggle/working'), Path('/kaggle/input')
assert WORK.is_dir(), 'Kaggle Cloud environment required'
REV = '6042f5bf5c8beac49784dbedba8034d93d713ccd'
with tempfile.TemporaryDirectory(prefix='w23_c1_') as ts:
    tmp = Path(ts)
    repo, meta_dir, vcache, sc = tmp / 'repo', tmp / 'pinns_meta', tmp / 'vcache', tmp / 'sc'
    subprocess.run(['git', 'init', '-q', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '-q', '--depth=1', 'https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git', REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
    env4 = {**os.environ, 'PYTHONPATH': str(repo), 'PYTHONDONTWRITEBYTECODE': '1', 'OMP_NUM_THREADS': '4', 'MKL_NUM_THREADS': '4'}
    joint_bin = next(MOUNT.rglob('joint_actions_all.bin'))
    heldout_bin = next(MOUNT.rglob('shortpose_heldout.bin'))
    map_bin = next(MOUNT.rglob('pinns_source_label_causal_maps.bin'))
    pose_npz = tmp / 'pose.npz'
    with zipfile.ZipFile(heldout_bin) as zf:
        pose_npz.write_bytes(zf.read('pose.npz'))
    subprocess.run([sys.executable, '-m', 'scripts.stage_pinns_metadata', '--output-dir', str(meta_dir)], cwd=repo, env=env4, check=True)
    out_zip = WORK / 'wave23_pinns_video_audit_and_6slot_ranker.zip'
    sheets_zip = WORK / 'pinns_blind_video_sheets.zip'
    subprocess.run([
        sys.executable, '-u', '-m', 'scripts.run_wave23_pinns_video_audit_and_6slot_ranker',
        '--meta-dir', str(meta_dir), '--pose-npz', str(pose_npz),
        '--joint-zip', str(joint_bin), '--causal-map-bin', str(map_bin),
        '--video-cache-dir', str(vcache), '--sheets-zip', str(sheets_zip),
        '--run-jrdb', '--scratch-dir', str(sc), '--zip-path', str(out_zip)
    ], cwd=repo, env=env4, check=True)
    for d in (out_zip, sheets_zip):
        if d.is_file():
            print('DONE:', d, d.stat().st_size, hashlib.sha256(d.read_bytes()).hexdigest(), flush=True)
